In [ ]:
import os
import sys

sys.path.insert(0, os.getcwd())
sys.path.insert(0, os.path.dirname(os.getcwd()))
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "DeepUnitMatch"))

import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
import umap.umap_ as umap
from tqdm import tqdm
import h5py

# from umap_plots import *
from testing.test import load_trained_model
from utils.param_fun import extract_Rwaveforms, save_waveforms_hdf5

umap_path = r"/Users/suyash/Projects/DeepUnitMatch/UMAPdata"

In [ ]:
# Forward pass the data through the network

data_root = r"/Volumes/UnionSine/DUM_DATA"
model_name = "original"
original_train_mice = [
    "AL031",
    "AL032",
    "AL036",
    "AV008",
    "CB015",
    "CB016",
    "CB017",
    "CB018",
    "CB020",
    "EB019",
]
additional_eight_mice = [
    "AV009",
    "AV015",
    "AV021",
    "AV049",
    "EB014",
    "FT033",
    "FT039",
    "JF084",
]
embedded_1, embedded_2, mouse, bc, depths = embed_data(data_root, model_name, original_train_mice, True, True)

np.save(os.path.join(umap_path, "first_half.npy"), embedded_1)
np.save(os.path.join(umap_path, "second_half.npy"), embedded_2)
pd.DataFrame(bc).to_csv(os.path.join(umap_path, "bombcell.csv"))
np.save(os.path.join(umap_path, "depths.npy"), np.array(depths))

In [ ]:
rWaveforms = []
all_labels = []
experiments = [
    r"\\zortex\Subjects\BZ008\2024-06-26\ephys\2024-06-26_BZ008_g0\2024-06-26_BZ008_g0_imec0",
    r"\\zortex\Subjects\BZ008\2024-06-25\ephys\2024-06-25_BZ008_g0\2024-06-25_BZ008_g0_imec0",
    r"\\zortex\Subjects\BZ008\2024-06-24\ephys\2024-06-24_BZ008_g0\2024-06-24_BZ008_g0_imec0",
]
for exp in experiments:
    labels = np.load(os.path.join(exp, "labels.npy"))
    for i, unit in enumerate(os.listdir(os.path.join(exp, "processed_waveforms"))):
        unit_path = os.path.join(exp, "processed_waveforms", unit)
        with h5py.File(unit_path, "r") as f:
            data = f["waveform"][()]
        if data.shape != (60, 30, 2):
            print(
                f"Expected Rwaveform shape to be (60, 30, 2) - instead got {data.shape}"
            )
        rWaveforms.append((data[:, :, 0] + data[:, :, 1]) * 0.5)
        all_labels.append(labels[i])
rWaveforms = np.array(rWaveforms)
all_labels = np.array(all_labels)

with torch.no_grad():
    model = load_trained_model(model_name)
    embedded = model(torch.tensor(rWaveforms)).detach().numpy()

In [ ]:
# n_neighbours = 5
# min_dist = 0.1
# n_components = 2
# metric = "euclidean"

# umap_embedding = umap.UMAP(n_neighbours, n_components, metric, min_dist=min_dist, random_state=0).fit_transform(0.5 * (embedded_1 + embedded_2), axis=0)
# np.save(os.path.join(umap_path, "UMAPembeddings.npy"), umap_embedding)

umap_embedding = np.load(os.path.join(umap_path, "UMAPembeddings.npy"))

In [ ]:
noclass = umap_embedding[-1543:][all_labels == 0]
plt.scatter(noclass[:, 0], noclass[:, 1], alpha=0.2, s=0.1, c="gray")
subset = umap_embedding[-1543:][all_labels > 0]
lab_subset = all_labels[all_labels > 0]
plt.scatter(subset[:, 0], subset[:, 1], alpha=1, c=lab_subset, s=1)
plt.colorbar()
plt.scatter(
    umap_embedding[:-1543, 0], umap_embedding[:-1543, 1], alpha=0.02, s=0.1, c="red"
)
plt.legend()
plt.savefig(os.path.join(umap_path, "UMAPplot.png"), dpi=500)

In [ ]:
# Do the UMAP
n_neighbours = 5
min_dist = 0.1
n_components = 2
metric = "euclidean"

umap_embedding = umap.UMAP(
    n_neighbours, n_components, metric, min_dist=min_dist, random_state=0
).fit_transform(embedded_1)
np.save(os.path.join(umap_path, "UMAPembeddings.npy"), umap_embedding)

In [ ]:
# Plot the UMAP embeddings
single_mouse = None

umap_embedding = np.load(os.path.join(umap_path, "UMAPembeddings.npy"))

fig, ax = plt.subplots(figsize=(11, 8), constrained_layout=False)
if single_mouse:
    labels = [
        original_train_mice[index]
        for index in mouse
        if original_train_mice[index] == single_mouse
    ]
    indices = [i for i in mouse if original_train_mice[i] == single_mouse]
    umap_to_plot = np.array(
        [
            umap_embedding[i, :]
            for i in range(len(mouse))
            if original_train_mice[mouse[i]] == single_mouse
        ]
    )
else:
    labels = [original_train_mice[index] for index in mouse]
    indices = mouse
    umap_to_plot = umap_embedding

scatter = ax.scatter(
    x=umap_to_plot[:, 0], y=umap_to_plot[:, 1], s=0.3, c=indices, label=labels
)
unique_labels = dict(zip(indices, labels))  # Remove duplicates
handles = [
    plt.Line2D(
        [],
        [],
        marker="o",
        linestyle="",
        markersize=5,
        color=scatter.cmap(scatter.norm(m)),
    )
    for m in unique_labels.keys()
]

ax.legend(handles, unique_labels.values())

p = r"C:\Users\suyash\UCL\DeepUnitMatch\random_figs"
# plt.savefig(os.path.join(p, "UMAP", "untrained", "all"))

In [ ]:
# Colour code by depth

single_mouse = "AV008"

depths = np.array(depths) / 1000

AV8depths = []
for i, m in enumerate(mouse):
    if m == 3:
        AV8depths.append(depths[i])

fig, ax = plt.subplots(figsize=(11, 8), constrained_layout=False)
if single_mouse:
    labels = [
        original_train_mice[index]
        for index in mouse
        if original_train_mice[index] == single_mouse
    ]
    indices = [i for i in mouse if original_train_mice[i] == single_mouse]
    umap_to_plot = np.array(
        [
            umap_embedding[i, :]
            for i in range(len(mouse))
            if original_train_mice[mouse[i]] == single_mouse
        ]
    )
else:
    labels = [original_train_mice[index] for index in mouse]
    indices = mouse
    umap_to_plot = umap_embedding

scatter = ax.scatter(
    x=umap_to_plot[:, 0],
    y=umap_to_plot[:, 1],
    s=0.3,
    c=AV8depths,
    cmap="viridis",
    label=labels,
)
cbar = plt.colorbar(scatter, ax=ax, pad=0.01, label="Depth (mm)")

In [ ]:
# Colour code UMAP by Bombcell output
bc_param = "waveformDuration_peakTrough"
# bc_param = "spatialDecaySlope"
# bc_param = "nSpikes"

umap_to_plot = umap_embedding

if bc_param == "rawAmplitude":
    df = pd.DataFrame(bc)
    df = df.loc[df["rawAmplitude"] < 500]
    mask = np.zeros(len(bc[bc_param]), dtype=bool)
    for i in df.index:
        mask[i] = True
    umap_to_plot = umap_embedding[mask]
    colours = df[bc_param].values
elif bc_param == "nSpikes":
    df = pd.DataFrame(bc)
    df = df.loc[df["nSpikes"] < 50000]
    mask = np.zeros(len(bc[bc_param]), dtype=bool)
    for i in df.index:
        mask[i] = True
    umap_to_plot = umap_embedding[mask]
    colours = df[bc_param].values
elif bc_param == "spatialDecaySlope":
    df = pd.DataFrame(bc)
    df = df.loc[df["spatialDecaySlope"] < 0]
    df = df.loc[df["spatialDecaySlope"] > -0.015]
    mask = np.zeros(len(bc[bc_param]), dtype=bool)
    for i in df.index:
        mask[i] = True
    umap_to_plot = umap_embedding[mask]
    colours = df[bc_param].values
else:
    umap_to_plot = umap_embedding
    colours = bc[bc_param]

fig, ax = plt.subplots(figsize=(11, 8), constrained_layout=False)
cmap = plt.cm.viridis
scatter = ax.scatter(
    x=umap_to_plot[:, 0],
    y=umap_to_plot[:, 1],
    s=0.3,  # Small point size
    c=colours,
    cmap=cmap,
    alpha=0.8,
)
cbar = plt.colorbar(scatter, ax=ax, pad=0.01)
cbar.set_label("Waveform Duration", fontsize=14)
ax.set_title("UMAP Visualisation Colored by Waveform Duration", fontsize=14)
ax.set_xlabel("UMAP Dimension 1", fontsize=14)
ax.set_ylabel("UMAP Dimension 2", fontsize=14)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

dir = r"C:\Users\suyash\UCL\DeepUnitMatch\random_figs\UMAP"
plt.tight_layout()
# plt.axis('off')
# plt.gca().set_position([0, 0, 1, 1])
plt.savefig(os.path.join(dir, f"{bc_param}.svg"), dpi=300, format="svg")
plt.rcParams["svg.fonttype"] = "none"
ax = plt.gca()
ax.spines[["right", "top"]].set_visible(False)
plt.savefig(os.path.join(dir, f"{bc_param}.png"), dpi=300, format="png")
plt.show()